# 01. Исходный вектор и валидационная выборка
Для извлечения векторов и генерации требуется GPU. В блоке запуска задаются имя, признак и каталог результатов на Google Drive.

In [ ]:
from hashlib import sha256
from pathlib import Path
from runpy import run_path
from subprocess import run

REPO_URL = "https://github.com/danilov-e/lora-activation-steering.git"
RUNS_DIR = Path("/content/drive/MyDrive/lora_activation_steering/runs")
RUN_NAME = "research_01"
TRAIT = "sycophancy"
GIT_REF = "main"

repo_id = sha256(REPO_URL.encode()).hexdigest()[:12]
source = Path("/content/lora-steering") / repo_id / "source"
if not source.exists():
    source.parent.mkdir(parents=True, exist_ok=True)
    run(["git", "clone", "--quiet", "--depth", "1", REPO_URL, str(source)], check=True)
else:
    run(["git", "-C", str(source), "fetch", "--quiet", "origin", "HEAD"], check=True)
    run(["git", "-C", str(source), "checkout", "--quiet", "--detach", "FETCH_HEAD"], check=True)

colab = run_path(str(source / "notebooks" / "colab.py"))
PROJECT, RUNS, COMMIT = colab["setup"](source, RUNS_DIR, RUN_NAME, TRAIT, ref=GIT_REF)

from src import judge as j, report as r, workflow as w
from IPython.display import Image, display
import pandas as pd

## 1. Настройки эксперимента
Сетка по слоям и α задаётся в профиле признака. Параметры можно изменить до создания запуска. Оценка по умолчанию использует GPT-6.1 Sol с уровнем Extra High.

In [ ]:
profile = w.read_json(PROJECT / "configs" / "research.json")[TRAIT]
CONFIG = w.default_config()
for key in ("layers", "alphas", "dev_repeats", "test_repeats"):
    CONFIG[key] = profile[key]

RUN = w.create_run(PROJECT, RUN_NAME, TRAIT, CONFIG, runs_dir=RUNS, git_commit=COMMIT)
DEV = RUN / "dev"
conditions = w.sweep_conditions(CONFIG)
expected = len(w.pairs(RUN, "dev")) * len(conditions) * CONFIG["dev_repeats"]
batch_size = CONFIG["judge"]["batch_size"]
print(RUN)
print(f"Судья: {CONFIG['judge']['model']}, reasoning={CONFIG['judge']['reasoning']}")
print(f"Dev: {expected} ответов, около {(expected + batch_size - 1) // batch_size} запросов")

## 2. Извлечение исходного вектора
Номера слоёв отсчитываются от нуля. Косинусное сходство между векторами двух половин выборки показывает устойчивость направления на этих данных.

In [ ]:
w.extract(RUN)
geometry, plot = r.extraction_report(RUN)
display(geometry.loc[geometry.layer.isin(CONFIG["layers"])].round(4))
display(Image(filename=str(plot)))

## 3. Генерация ответов
Ответы сохраняются на Google Drive по мере генерации. После прерывания ячейка продолжает работу с сохранённого состояния.

In [ ]:
w.generate(RUN, "dev", DEV, conditions, repeats=CONFIG["dev_repeats"])

## 4. Оценка через OpenAI
Ячейка выполняет платные запросы. Ключ `OPENAI_API_KEY` считывается из Colab Secrets или вводится скрыто. Успешно обработанные пакеты сохраняются.

In [ ]:
colab["api_key"]()
display(j.judge(DEV, TRAIT, **CONFIG["judge"]))

## 5. Анализ валидационной выборки
Таблицы и графики показывают изменение целевого признака и связности относительно базовой модели.

In [ ]:
frame = r.scored(DEV, TRAIT)
summary, effects, figures = r.dev_report(frame, TRAIT, DEV)
display(summary.sort_values("target_mean", ascending=False).round(2))
display(effects.round(2))
for plot in figures:
    display(Image(filename=str(plot)))

candidates = summary[summary.condition.ne("base:baseline")].dropna(subset=["target_mean"])
if not candidates.empty:
    condition = candidates.sort_values("target_mean", ascending=False).iloc[0].condition
    display(r.examples(frame, TRAIT, condition))

## 6. Фиксация конфигураций
В список `CHOICES` вносятся две или три разные комбинации из таблицы. Каждая запись содержит ключи `name`, `layer` и `alpha`. Например, `{"name": "c1", "layer": 18, "alpha": 2.5}`. Имена назначаются последовательно от `c1`. После сохранения можно перейти к ноутбуку 02.

In [ ]:
CHOICES = []
if CHOICES:
    display(w.save_choices(RUN, CHOICES))
else:
    print("Для перехода к ноутбуку 02 необходимо сохранить список конфигураций.")